# Time-series Forecasting on Household Power Consumption
This notebook covers exploratory data analysis for this project.

The general outline follows these steps.
1. Ingest data, reading '?' as NA
2. Interpolate missing data
3. Aggregate to daily mean
4. EDA
5. Stationarity
6. Train/test split

---

## Step 0: Imports and libraries

In [1]:
# ----------------------------------------------
# Step 0: Import libraries set up ennvironment
# ----------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import pmdarima
import tensorflow as tf


## Step 1: Data ingest

In [25]:
df = pd.read_csv('../data/household_power_consumption.txt',
                sep=';', na_values=['?'],
                low_memory=False)
df.index = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
df.index.name = 'datetime'
df.drop(columns=['Date', 'Time'], inplace=True)

# rename to more descriptive columns
df.columns = ["power", "reactive_power", "voltage", "current", "sub_meter_kitchen", "sub_meter_laundry", "sub_meter_heating"]

## Step 2: Handle Missing Values

**Decision: Linear interpolation at the minute level**

Missing values appear as `?` (read as `NaN`) and cluster in short gaps rather than extended outages. Three options were considered:

| Approach | Problem |
|---|---|
| Drop rows | Biases daily aggregates downward — a day with 60 missing minutes loses ~4% of observations before averaging |
| Forward-fill | Over-represents the last known state; assumes usage was flat during the gap, which is unlikely for a continuous signal |
| **Linear interpolation** ✓ | Assumes a smooth transition between known values — the most defensible assumption for a continuous physical measurement |

`asfreq('min')` first ensures every minute-level timestamp exists in the index (inserting `NaN` rows for any completely absent timestamps). `interpolate(method='linear')` then fills all `NaN` values by connecting known neighbors with a straight line.

In [26]:
print(f"Missing values before interpolation:\n{df.isnull().sum()}\n")

df = df.asfreq('min')
df = df.interpolate(method='linear')

print(f"Missing values after interpolation:\n{df.isnull().sum()}")

Missing values before interpolation:
power                25979
reactive_power       25979
voltage              25979
current              25979
sub_meter_kitchen    25979
sub_meter_laundry    25979
sub_meter_heating    25979
dtype: int64

Missing values after interpolation:
power                0
reactive_power       0
voltage              0
current              0
sub_meter_kitchen    0
sub_meter_laundry    0
sub_meter_heating    0
dtype: int64


## Step 3: Aggregation

In [27]:
mean_cols = ["power", "reactive_power", "voltage", "current"]
sum_cols  = ["sub_meter_kitchen", "sub_meter_laundry", "sub_meter_heating"]

daily = pd.concat([
    df[mean_cols].resample('D').mean(),
    df[sum_cols].resample('D').sum(),
], axis=1)

daily.head()

,power,reactive_power,voltage,current,sub_meter_kitchen,sub_meter_laundry,sub_meter_heating
datetime,,,,,,,
2006-12-16,3.053475,0.088187,236.243763,13.082828,0.0,546.0,4926.0
2006-12-17,2.354486,0.156949,240.087028,9.999028,2033.0,4187.0,13341.0
2006-12-18,1.530435,0.112356,241.231694,6.421667,1063.0,2621.0,14018.0
2006-12-19,1.157079,0.104821,241.999313,4.926389,839.0,7602.0,6197.0
2006-12-20,1.545658,0.111804,242.308062,6.467361,0.0,2648.0,14063.0


## Step 4: EDA